# Waiter-month model tuning

Tune **Isolation Forest**, **One-Class SVM**, and **LOF** on waiter–month features.

Each model uses its own feature list (same as `models/waiter_month_models.py`: **iso**, **ocsvm**, **lof**).

**Objective:** maximize **recall@100** — among all rows, take the top 100 by anomaly score (higher = more anomalous); **recall@100** is the fraction of known frauds that fall in that top set. Same definition as in `models/fit_and_evaluate.py`.

In [1]:
import sys
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

_cwd = Path.cwd()
if _cwd.name in ("A_tuning", "models_tuning"):
    ROOT = _cwd.parent
else:
    ROOT = _cwd
sys.path.insert(0, str(ROOT))

from config import load_data
from models.scaling import scale_features

# Same dict as models/waiter_month_models.WAITER_MONTH_FEATURES_BY_MODEL (avoid importing that module).
WAITER_MONTH_FEATURES_ISO = [
    "trn_per_person_norm",
    "top1_client_trn",
    "bonusses_accum",
    "trn_per_person",
    "top1_client_share_norm",
    "top1_client_trn_diff_next",
    "top1_client_trn_diff_prev",
    "mean_check",
    "bonusses_used",
    "top1_client_trn_perc_diff_next",
]
WAITER_MONTH_FEATURES_OCSVM = [
    "trn_per_person_norm",
    "trn_per_person_perc_diff_next",
    "top1_client_share_norm",
    "bonusses_accum_diff_prev",
    "share_loyal_trn",
]
WAITER_MONTH_FEATURES_LOF = [
    "trn_per_day_norm",
    "bonusses_accum_diff_next",
    "trn_per_person_perc_diff_next",
    "trn_per_person_norm_perc_diff_next",
    "trn_per_person",
    "unique_persons_diff_next",
    "trn_per_person_norm",
    "bonusses_accum",
    "mean_check",
    "top1_client_share_norm",
    "unique_clients_per_day",
    "unique_clients_per_day_diff_prev",
    "top1_client_trn_diff_next",
    "bonusses_trn",
    "unique_clients_per_day_perc_diff_prev",
    "trn_per_person_norm_perc_diff_prev",
    "share_bonusses_trn",
    "top1_client_trn_diff_prev",
    "share_loyal_trn",
]

MIN_NUM_OF_TRN = 8
NUM_OF_TRN = 1
PLACE_NUM_OF_WAITERS = 1
MIN_WORKING_DAYS = 2
NUM_OF_TRN_MONTH = 10
EXCLUDE_FRAUD_FROM_TRAINING = True
MAX_OCSVM_TRAIN = 4000
RANDOM_STATE = 42

_, _, _, waiter_month_data, _ = load_data(
    num_of_trn=NUM_OF_TRN,
    place_num_of_waiters=PLACE_NUM_OF_WAITERS,
    min_working_days=MIN_WORKING_DAYS,
    num_of_trn_month=NUM_OF_TRN_MONTH,
)
waiter_month_data = waiter_month_data[
    waiter_month_data["num_of_trn"] >= MIN_NUM_OF_TRN
].copy()
y_fraud = waiter_month_data["is_fraud"].astype(int).values
n_fraud = int(y_fraud.sum())
n_total = len(waiter_month_data)
print(f"Samples: {n_total} | Known fraud waiter-months: {n_fraud}")
print(f"Features ISO ({len(WAITER_MONTH_FEATURES_ISO)}): {WAITER_MONTH_FEATURES_ISO}")
print(f"Features OCSVM ({len(WAITER_MONTH_FEATURES_OCSVM)}): {WAITER_MONTH_FEATURES_OCSVM}")
print(f"Features LOF ({len(WAITER_MONTH_FEATURES_LOF)}): {WAITER_MONTH_FEATURES_LOF}")

train_mask = y_fraud == 0
fit_subset = waiter_month_data.loc[train_mask] if EXCLUDE_FRAUD_FROM_TRAINING else None

X_fit = {}
X_eval = {}
for key, feats in (
    ("iso", WAITER_MONTH_FEATURES_ISO),
    ("ocsvm", WAITER_MONTH_FEATURES_OCSVM),
    ("lof", WAITER_MONTH_FEATURES_LOF),
):
    if EXCLUDE_FRAUD_FROM_TRAINING:
        X_fit_df, X_eval_df = scale_features(
            data=waiter_month_data,
            features=feats,
            scaler_type="standard",
            fit_data=fit_subset,
        )
        X_fit[key] = X_fit_df.to_numpy(dtype=np.float64, copy=False)
        X_eval[key] = X_eval_df.to_numpy(dtype=np.float64, copy=False)
    else:
        X_full = scale_features(
            data=waiter_month_data,
            features=feats,
            scaler_type="standard",
        )
        arr = X_full.to_numpy(dtype=np.float64, copy=False)
        X_fit[key] = arr
        X_eval[key] = arr

n_fit = len(X_fit["iso"])

K_OPT = 100  # optimize recall @ this rank cutoff


def recall_at_k(anomaly_scores: np.ndarray, fraud_binary: np.ndarray, k: int = K_OPT) -> float:
    """Higher anomaly_scores = more anomalous (same convention as fit_and_evaluate)."""
    fraud_binary = np.asarray(fraud_binary).astype(int)
    n_fraud = int(fraud_binary.sum())
    if n_fraud == 0:
        return float("nan")
    order = np.argsort(-anomaly_scores)
    fraud_mask = fraud_binary.astype(bool)
    top_k = order[: min(k, len(order))]
    return float(fraud_mask[top_k].sum() / n_fraud)


def lof_n_neighbors(requested: int, n_fit_samples: int) -> int:
    return int(min(max(5, requested), max(1, n_fit_samples - 1)))

Samples: 18049 | Known fraud waiter-months: 28
Features ISO (10): ['trn_per_person_norm', 'top1_client_trn', 'bonusses_accum', 'trn_per_person', 'top1_client_share_norm', 'top1_client_trn_diff_next', 'top1_client_trn_diff_prev', 'mean_check', 'bonusses_used', 'top1_client_trn_perc_diff_next']
Features OCSVM (5): ['trn_per_person_norm', 'trn_per_person_perc_diff_next', 'top1_client_share_norm', 'bonusses_accum_diff_prev', 'share_loyal_trn']
Features LOF (19): ['trn_per_day_norm', 'bonusses_accum_diff_next', 'trn_per_person_perc_diff_next', 'trn_per_person_norm_perc_diff_next', 'trn_per_person', 'unique_persons_diff_next', 'trn_per_person_norm', 'bonusses_accum', 'mean_check', 'top1_client_share_norm', 'unique_clients_per_day', 'unique_clients_per_day_diff_prev', 'top1_client_trn_diff_next', 'bonusses_trn', 'unique_clients_per_day_perc_diff_prev', 'trn_per_person_norm_perc_diff_prev', 'share_bonusses_trn', 'top1_client_trn_diff_prev', 'share_loyal_trn']


### Isolation Forest

Grid over `n_estimators`, `contamination`, `max_samples`. Uses **ISO** feature set. Metric: **recall@100** from anomaly scores (`-score_samples`).

In [2]:
rows = []

for n_est, cont, ms in product(
    [100, 200, 500],
    [0.0005],
    [1.0, 0.8],
):
    try:
        iso = IsolationForest(
            n_estimators=n_est,
            contamination=cont,
            max_samples=ms,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        iso.fit(X_fit["iso"])
        scores = -iso.score_samples(X_eval["iso"])
        rows.append(
            {
                "model": "IsolationForest",
                "recall@100": recall_at_k(scores, y_fraud),
                "n_estimators": n_est,
                "contamination": cont,
                "max_samples": ms,
            }
        )
    except Exception as exc:
        rows.append(
            {
                "model": "IsolationForest",
                "recall@100": np.nan,
                "n_estimators": n_est,
                "contamination": cont,
                "max_samples": ms,
                "error": str(exc),
            }
        )

print(
    f"Isolation Forest: {sum(1 for r in rows if r['model'] == 'IsolationForest')} fits, "
    f"rows total = {len(rows)}"
)

Isolation Forest: 6 fits, rows total = 6


### One-Class SVM

Subsample training to `MAX_OCSVM_TRAIN` when needed (same as `fit_and_evaluate`). Uses **OCSVM** feature set. Metric: **recall@100** from `-decision_function`.

In [3]:
rng = np.random.default_rng(RANDOM_STATE)
n_fit_ocsvm = len(X_fit["ocsvm"])
if n_fit_ocsvm > MAX_OCSVM_TRAIN:
    idx_sub = rng.choice(n_fit_ocsvm, size=MAX_OCSVM_TRAIN, replace=False)
    X_ocsvm_fit = X_fit["ocsvm"][idx_sub]
else:
    X_ocsvm_fit = X_fit["ocsvm"]

for nu, gamma in product(
    [0.0005, 0.001, 0.005, 0.01, 0.05, 0.1],
    ["scale", "auto"],
):
    try:
        ocsvm = OneClassSVM(kernel="rbf", nu=nu, gamma=gamma)
        ocsvm.fit(X_ocsvm_fit)
        scores = -ocsvm.decision_function(X_eval["ocsvm"])
        rows.append(
            {
                "model": "OneClassSVM",
                "recall@100": recall_at_k(scores, y_fraud),
                "nu": nu,
                "gamma": gamma,
            }
        )
    except Exception as exc:
        rows.append(
            {
                "model": "OneClassSVM",
                "recall@100": np.nan,
                "nu": nu,
                "gamma": gamma,
                "error": str(exc),
            }
        )

print(
    f"One-Class SVM: {sum(1 for r in rows if r['model'] == 'OneClassSVM')} fits, "
    f"rows total = {len(rows)}"
)

One-Class SVM: 12 fits, rows total = 18


### LOF (Local Outlier Factor)

`novelty=True`: fit on `X_fit`, score on `X_eval`. Uses **LOF** feature set. Metric: **recall@100** from `-score_samples`. Sweeps many **`n_neighbors`** values (each clamped to `[5, n_fit - 1]`).

In [4]:
n_fit_lof = len(X_fit["lof"])
for nn_req, cont in product(
    [5, 10, 15, 20, 30, 40, 50, 75, 100, 150, 200, 300, 500, 750, 1000, 2000, 5000, 10000],
    [0.01],
):
    nn = lof_n_neighbors(nn_req, n_fit_lof)
    try:
        lof = LocalOutlierFactor(
            n_neighbors=nn,
            contamination=cont,
            metric="minkowski",
            p=2,
            novelty=True,
        )
        lof.fit(X_fit["lof"])
        scores = -lof.score_samples(X_eval["lof"])
        rows.append(
            {
                "model": "LOF",
                "recall@100": recall_at_k(scores, y_fraud),
                "n_neighbors_requested": nn_req,
                "n_neighbors": nn,
                "contamination": cont,
            }
        )
    except Exception as exc:
        rows.append(
            {
                "model": "LOF",
                "recall@100": np.nan,
                "n_neighbors_requested": nn_req,
                "n_neighbors": nn,
                "contamination": cont,
                "error": str(exc),
            }
        )

print(f"LOF: {sum(1 for r in rows if r['model'] == 'LOF')} fits, rows total = {len(rows)}")

LOF: 18 fits, rows total = 36


### Summary

Build `tuning_results` from all `rows` and rank by **`recall@100`**. Run the three model code cells in order (Isolation Forest first — it initializes `rows`).

In [5]:
tuning_results = pd.DataFrame(rows)
metric = "recall@100"
print(f"Finished {len(tuning_results)} fits (metric: {metric}).\n")

for model_name in ["IsolationForest", "OneClassSVM", "LOF"]:
    sub = tuning_results[tuning_results["model"] == model_name].sort_values(
        metric, ascending=False, na_position="last"
    )
    print(f"=== Top settings: {model_name} (by {metric}) ===")
    print(sub.head(8).to_string(index=False))
    print()

best = tuning_results.sort_values(metric, ascending=False, na_position="last").iloc[0]
print(f"Best overall by {metric}:")
print(best.to_string())

Finished 36 fits (metric: recall@100).

=== Top settings: IsolationForest (by recall@100) ===
          model  recall@100  n_estimators  contamination  max_samples  nu gamma  n_neighbors_requested  n_neighbors
IsolationForest    0.285714         500.0         0.0005          1.0 NaN   NaN                    NaN          NaN
IsolationForest    0.250000         100.0         0.0005          1.0 NaN   NaN                    NaN          NaN
IsolationForest    0.250000         200.0         0.0005          1.0 NaN   NaN                    NaN          NaN
IsolationForest    0.250000         500.0         0.0005          0.8 NaN   NaN                    NaN          NaN
IsolationForest    0.214286         100.0         0.0005          0.8 NaN   NaN                    NaN          NaN
IsolationForest    0.214286         200.0         0.0005          0.8 NaN   NaN                    NaN          NaN

=== Top settings: OneClassSVM (by recall@100) ===
      model  recall@100  n_estimators  cont